# W02a - Topic Modeling Demonstration & Salary Predictions (F(x) Model)

In [28]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import gensim
import time
from text_operations import manipulate_text, translate_to_english
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
num_topics = 25    # It is crucial to set the number of topics to be created for topic modeling
salaries = pd.read_csv('Salaries_v4_202412161100.csv')
# salaries = pd.read_csv('Salaries_v3_202410141100.csv')
# salaries = pd.read_csv('Salaries_v3_202408271150.csv')
# salaries = pd.read_csv('Salaries_v3_202408021730.csv')
# salaries = pd.read_csv('Salaries_v3_202405031235.csv')
# salaries = pd.read_csv('Salaries_v3_202404091315.csv')
# salaries = pd.read_csv('Salaries_v3_202402121435.csv')
# salaries = pd.read_csv('Salaries_v3_202401081515.csv')
countries_coef = pd.read_csv('countries_salary_coef.csv')
salaries = salaries.drop_duplicates('id').reset_index()
salaries = salaries.drop('index',axis=1)

In [29]:
# Remove the specific rows that can cause inconsistency to the salary dataset
# Salaries_v3_202410141100.csv
# salaries = salaries.drop(17343,axis=0)
## Salaries_v4_202412161100.csv
salaries = salaries.drop([14255,8691,8931],axis=0)

## Dataset Information

In [30]:
# Number of rows (salaries) and columns
print("SALARY DATASET SHAPE -> ROWS: {}  COLUMNS: {}".format(salaries.shape[0], salaries.shape[1]))

SALARY DATASET SHAPE -> ROWS: 19554  COLUMNS: 16


In [31]:
# First few rows of the dataset
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000


In [32]:
# General info about the salaries dataset
salaries.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19554 entries, 0 to 19556
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 19554 non-null  object 
 1   found_country      19554 non-null  object 
 2   title              19542 non-null  object 
 3   position           19554 non-null  object 
 4   employment_type    12397 non-null  object 
 5   company            16936 non-null  object 
 6   company_score      15739 non-null  float64
 7   edu_degrees        19000 non-null  object 
 8   edu_degrees_major  18623 non-null  object 
 9   working_year       19554 non-null  int64  
 10  education_score    14956 non-null  float64
 11  ms_counts          19554 non-null  int64  
 12  skill_counts       19554 non-null  int64  
 13  main_skills        19554 non-null  object 
 14  skills             19554 non-null  object 
 15  amount_usd         19554 non-null  int64  
dtypes: float64(2), int64(4

In [33]:
# Stats for numerical columns of salaries dataset
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000


In [34]:
# Observe the minimum and maximum working years in the whole dataset
print("MIN WORKING YEAR: {} / MAX WORKING YEAR: {}".format(
    salaries['working_year'].min(), salaries['working_year'].max()))

MIN WORKING YEAR: 0 / MAX WORKING YEAR: 57


In [35]:
# Percentage values of gross salaries (for confidence interval) 
print("#### CONFIDENCE INTERVALS ####")
print("95% ->", np.percentile(salaries['amount_usd'], [2.5, 97.5]))
print("98% ->", np.percentile(salaries['amount_usd'], [1.0, 99.0]))
print("99% ->", np.percentile(salaries['amount_usd'], [0.5, 99.5]))

#### CONFIDENCE INTERVALS ####
95% -> [ 47074.25 350000.  ]
98% -> [ 42000. 400000.]
99% -> [ 38187. 446175.]


In [36]:
# Countries and their salary coefficients (first 20 rows)
countries_coef[:20]

,tr_name,en_name,salary_coefficient
0,Amerika Birleşik Devletleri,United States,1.000
1,Kanada,Canada,0.769
2,Monako,Monaco,0.766
3,Lüksemburg,Luxembourg,0.766
4,İrlanda,Ireland,0.765
5,İsrail,Israel,0.763
6,İzlanda,Iceland,0.750
7,İsviçre,Switzerland,0.740
8,Malta,Malta,0.735
9,Tayland,Thailand,0.729


## Preprocessing

In [38]:
# Process the null titles and companies to become empty strings
salaries['title'] = salaries['title'].apply(lambda x: '' if type(x) == float else x)
salaries['company'] = salaries['company'].apply(lambda x: '' if type(x) == float else x)

In [41]:
# Convert all textual values into list vectors
# All characters must be lowercase and several specific characters and substring should be converted into meaningful ones
salaries_text_processed = (salaries['title'] + ' ' + salaries['main_skills'] + ' ' + salaries['skills']).str.lower()
s_time = time.time()
# for i in range(len(salaries_text_processed)):
#     salaries_text_processed.iloc[i] = salaries_text_processed.iloc[i].replace(' sr ',' senior ') \
#         .replace(' sr. ',' senior ').replace('co.','company').replace('front-end','frontend') \
#         .replace('back-end','backend').replace('object-oriented','object oriented').replace('t-sql','tsql') \
#         .replace('pl/sql','plsql').replace('.js','js').replace('tcp/ip','tcpip').replace('ui/ux','uiux') \
#         .replace('ci/cd','cicd').replace('html 5','html5').replace(' ii ',' ').replace(' at ',' ').replace(' a ',' ') \
#         .replace(',','').replace('.','dot').replace('#','sharp').replace('+','plus').replace('/','slash') \
#         .replace('-','dash').replace('&','and').replace('(','').replace(')','').replace('[','').replace(']','') \
#         .replace('@','').replace('|','').replace('  ',' ')
for i in range(len(salaries_text_processed)):
    text = salaries_text_processed.iloc[i]
    manipulated_text = manipulate_text(text)
    translated_text = translate_to_english(manipulated_text)
    salaries_text_processed.iloc[i] = translated_text
salaries_text_processed = salaries_text_processed.apply(lambda x: x.split(' '))
print(">>> TOTAL TEXT PREPROCESSING TIME: {:.4f} seconds".format(time.time()-s_time))

>>> TOTAL TEXT PREPROCESSING TIME: 5.3216 seconds


In [44]:
# Check the list vectors of 5 randomly selected rows
for i in range(5):
    rand_row = random.randint(0,len(salaries_text_processed)-1)
    print("### ROW {} ###\n{}".format(rand_row,salaries_text_processed.iloc[rand_row]))

### ROW 3955 ###
['software', 'engineer', 'bloomberg', 'lp', 'dotnet', 'agile', 'analytical', 'skills', 'android', 'programming', 'angular', 'cplusplus', 'facebook', 'financial', 'management', 'google', 'analytics', 'javascript', 'laravel', 'machine', 'learning', 'marketing', 'object', 'oriented', 'programming', 'programming', 'project', 'management', 'python', 'react', 'rest', 'sales', 'marketing', 'scrum', 'web', 'applications', 'web', 'design', 'web', 'development', 'dotnet', 'framework', 'agile', 'methodologies', 'analytical', 'skills', 'android', 'development', 'angularjs', 'cplusplus', 'facebook', 'marketing', 'google', 'analytics', 'javascript', 'laravel', 'machine', 'learning', 'objectoriented', 'programming', 'oop', 'project', 'management', 'python', 'reactjs', 'responsive', 'web', 'design', 'restful', 'webservices', 'scrum', 'web', 'applications', 'web', 'development']
### ROW 5639 ###
['leasing', 'consultant', 'cortland', 'customer', 'service', 'deadline', 'management', 'ehr

In [49]:
# Create a dictionary, containing the number of times a word appears in the training set
salaries_text_processed_sr = pd.Series(salaries_text_processed)
s_time = time.time()
dictionary = gensim.corpora.Dictionary(salaries_text_processed_sr)
first_words = []
count = 0
# The first 75 words are shown in the output below
print("#### {} WORDS FOUND IN THE DICTIONARY ####".format(len(dictionary)))
print("------------- THE FIRST 75 WORDS -------------")
for k, v in dictionary.iteritems():
    first_words.append(v)
    count += 1
    if count > 74:
        break
for i in range(25):
    print("{:3} - {:24} | {:3} - {:24} | {:3} - {:24}".format(
        i, first_words[i], i+25, first_words[i+25], i+50, first_words[i+50]))
print("TOTAL DICTIONARY CREATION TIME: {:.4f} seconds".format(time.time()-s_time))

#### 15375 WORDS FOUND IN THE DICTIONARY ####
------------- THE FIRST 75 WORDS -------------
  0 - ajax                     |  25 - sheets                   |  50 - techniques              
  1 - audible                  |  26 - software                 |  51 - training                
  2 - c                        |  27 - sql                      |  52 - trends                  
  3 - cascading                |  28 - style                    |  53 - website                 
  4 - cplusplus                |  29 - xml                      |  54 - agile                   
  5 - css                      |  30 - associate                |  55 - analysis                
  6 - data                     |  31 - building                 |  56 - api                     
  7 - database                 |  32 - communication            |  57 - attention               
  8 - developer                |  33 - converse                 |  58 - automated               
  9 - development              |  

## Bag of Words

In [50]:
# For each document, create an individual dictionary reporting how many words and how many times those words appear.
# Save this to 'bow_corpus'
bow_corpus = [dictionary.doc2bow(s) for s in salaries_text_processed_sr]

In [53]:
# Preview bag of words for the randomly selected row
rand_row = random.randint(0,len(bow_corpus)-1)
print("### ROW", rand_row, "###")
print(salaries_text_processed_sr[rand_row])
bow_doc_selected = bow_corpus[rand_row]
print(bow_doc_selected)
for i in range(len(bow_doc_selected)):
    print("Word {} (\"{}\") appears {} {}.".format(
        bow_doc_selected[i][0], dictionary[bow_doc_selected[i][0]], bow_doc_selected[i][1], 
        "times" if bow_doc_selected[i][1] > 1 else "time"))

### ROW 8715 ###
['senior', 'data', 'analyst', 'data', 'analytics', 'data', 'science', 'business', 'intelligence', 'azure', 'business', 'intelligence', 'data', 'analysis', 'data', 'engineering', 'data', 'visualization', 'etl', 'google', 'analytics', 'microsoft', 'excel', 'microsoft', 'powerbi', 'microsoft', 'sql', 'server', 'mysql', 'tableau', 'visual', 'azure', 'databricks', 'business', 'analytics', 'business', 'intelligence', 'bi', 'data', 'analysis', 'data', 'visualization', 'extract', 'transform', 'load', 'etl', 'google', 'analytics', 'google', 'bigquery', 'google', 'data', 'studio', 'microsoft', 'excel', 'microsoft', 'powerbi', 'microsoft', 'sql', 'server', 'mysql', 'tableau']
[(26, 1), (34, 2), (37, 1), (38, 1), (40, 6), (41, 1), (45, 3), (46, 6), (47, 2), (49, 1), (50, 1), (52, 1), (82, 1), (176, 1), (185, 1), (201, 1), (210, 5), (251, 1), (300, 1), (319, 7), (321, 2), (323, 3), (342, 1), (452, 1), (461, 1), (539, 1), (540, 1), (621, 2), (671, 1), (800, 1), (849, 1), (1449, 1), 

## TF-IDF

In [55]:
# Create TF-IDF model object using gensim.models.TfidfModel on 'bow_corpus' and save it to 'tfidf'
# Then apply transformation to the entire corpus and call it 'corpus_tfidf'
from gensim import corpora, models
tfidf = models.TfidfModel(bow_corpus)
corpus_tfidf = tfidf[bow_corpus]

In [57]:
# Preview TF-IDF scores for the randomly selected row
rand_row = random.randint(0,len(bow_corpus)-1)
print("### ROW {} ###".format(rand_row))
print(salaries_text_processed_sr[rand_row])
print(corpus_tfidf[rand_row])

### ROW 4512 ###
['digital', 'merchandiser', 'analytical', 'skills', 'communication', 'ecommerce', 'facebook', 'forecasting', 'merchandising', 'negotiation', 'pr', 'skills', 'product', 'development', 'product', 'knowledge', 'public', 'relations', 'retail', 'experience', 'retail', 'trends', 'social', 'media', 'social', 'media', 'platforms', 'teamwork', 'training', 'visual', 'analytical', 'skills', 'apparel', 'communication', 'ecommerce', 'facebook', 'fashion', 'fashion', 'buying', 'fashion', 'design', 'footwear', 'industry', 'training', 'luxury', 'goods', 'merchandise', 'planning', 'merchandising', 'negotiation', 'product', 'development', 'public', 'relations', 'retail', 'retail', 'buying', 'social', 'media', 'styling', 'teamwork', 'trend', 'trend', 'analysis', 'trend', 'forecasting', 'visual', 'merchandising', 'wholesale']
[(9, 0.04751387039440324), (32, 0.07508198102847027), (37, 0.050541386988054604), (45, 0.29812852174885246), (48, 0.09061659972198509), (51, 0.11955452860450007), (5

## Run LDA using Bag of Words

In [59]:
# Train the LDA model with BoW using the related method from gensim and save it to 'lda_model_bow'
s_time = time.time()
lda_model_bow = gensim.models.LdaModel(bow_corpus, num_topics=num_topics, id2word=dictionary, passes=3, random_state=42)
print("NUMBER OF TOPICS: {} | LDA using BoW lasted {:.3f} seconds.".format(num_topics, time.time() - s_time))

NUMBER OF TOPICS: 25 | LDA using BoW lasted 19.063 seconds.


In [61]:
# For each topic, explore the words occurring in that topic and its relative weights
# print(dir(lda_model_bow))
print("TOPIC --> WORDS")
for idx, topic in lda_model_bow.print_topics(-1):
    print('{:5} --> {}'.format(idx, topic))

TOPIC --> WORDS
    0 --> 0.098*"retail" + 0.083*"sales" + 0.081*"inventory" + 0.070*"customer" + 0.065*"management" + 0.047*"service" + 0.035*"trends" + 0.030*"merchandising" + 0.028*"experience" + 0.019*"support"
    1 --> 0.163*"microsoft" + 0.113*"management" + 0.065*"office" + 0.050*"excel" + 0.041*"word" + 0.041*"powerpoint" + 0.035*"customer" + 0.035*"leadership" + 0.035*"service" + 0.028*"research"
    2 --> 0.070*"medical" + 0.062*"care" + 0.062*"patient" + 0.060*"healthcare" + 0.058*"maintenance" + 0.029*"health" + 0.028*"education" + 0.026*"clinical" + 0.024*"emergency" + 0.020*"life"
    3 --> 0.219*"management" + 0.074*"project" + 0.066*"business" + 0.059*"analysis" + 0.041*"financial" + 0.034*"product" + 0.031*"process" + 0.027*"development" + 0.022*"improvement" + 0.020*"planning"
    4 --> 0.072*"management" + 0.065*"skills" + 0.063*"communication" + 0.052*"problem" + 0.051*"solving" + 0.035*"leadership" + 0.024*"customer" + 0.022*"teamwork" + 0.022*"analytical" + 0.020

## Run LDA using TF-IDF

In [62]:
# Train the LDA model with TF-IDF using the related method from gensim and save it to 'lda_model_tfidf'
s_time = time.time()
lda_model_tfidf = gensim.models.LdaModel(corpus_tfidf, num_topics=num_topics, id2word=dictionary, passes=3, random_state=42)
print("NUMBER OF TOPICS: {} | LDA using TF-IDF lasted {:.3f} seconds.".format(num_topics, time.time() - s_time))

NUMBER OF TOPICS: 25 | LDA using TF-IDF lasted 13.117 seconds.


In [64]:
# For each topic, explore the words occurring in that topic and its relative weight
print("TOPIC --> WORDS")
for idx, topic in lda_model_tfidf.print_topics(-1):
    print('{:5} --> {}'.format(idx, topic))

TOPIC --> WORDS
    0 --> 0.026*"valuation" + 0.016*"qlikview" + 0.015*"after" + 0.014*"informatica" + 0.014*"effects" + 0.013*"calculus" + 0.013*"stochastic" + 0.013*"ssis" + 0.012*"ssrs" + 0.011*"de"
    1 --> 0.032*"insurance" + 0.024*"risk" + 0.022*"finance" + 0.020*"banking" + 0.013*"investments" + 0.013*"financial" + 0.010*"corporate" + 0.009*"analysis" + 0.009*"management" + 0.009*"investment"
    2 --> 0.024*"career" + 0.016*"academic" + 0.016*"storage" + 0.015*"advising" + 0.012*"juniper" + 0.012*"student" + 0.011*"schools" + 0.011*"guidance" + 0.010*"counseling" + 0.010*"entity"
    3 --> 0.033*"management" + 0.031*"process" + 0.029*"business" + 0.024*"improvement" + 0.024*"project" + 0.017*"analysis" + 0.015*"product" + 0.015*"program" + 0.015*"requirements" + 0.014*"autocad"
    4 --> 0.025*"resolution" + 0.020*"conflict" + 0.019*"advocacy" + 0.014*"instagram" + 0.013*"deliveroo" + 0.012*"adaptable" + 0.012*"easily" + 0.012*"haskell" + 0.010*"twitter" + 0.010*"app"
    5 --

## Test LDA Bag of Words & LDA TF-IDF

In [72]:
rand_row = random.randint(0,len(bow_corpus)-1)
print("CHOSEN ROW: {}".format(rand_row))
print("VECTOR >>> {}".format(salaries_text_processed_sr[rand_row]))
print("BAG OF WORDS:", bow_corpus[rand_row])
print("---------- BoW ----------")
scores_bow = sorted(lda_model_bow[bow_corpus[rand_row]], key=lambda tup: -1*tup[1])
print("SCORE                | TOPIC")
for index, score in scores_bow:
    print("{:<20} | {} --> {}".format(score, index, lda_model_bow.print_topic(index, 10)))
print("---------- TF-IDF ----------")
scores_tfidf = sorted(lda_model_tfidf[bow_corpus[rand_row]], key=lambda tup: -1*tup[1])
print("SCORE                | TOPIC")
for index, score in scores_tfidf:
    print("{:<20} | {} --> {}".format(score, index, lda_model_tfidf.print_topic(index, 10)))

CHOSEN ROW: 12992
VECTOR >>> ['sales', 'associate', 'nordstrom', 'fashion', 'design', 'merchandising', 'basic', 'math', 'skills', 'customer', 'service', 'design', 'skills', 'merchandising', 'microsoft', 'excel', 'microsoft', 'word', 'photoshop', 'product', 'development', 'product', 'knowledge', 'retail', 'trends', 'sales', 'sales', 'marketing', 'sales', 'strategy', 'sales', 'techniques', 'user', 'interface', 'design', 'visual', 'month', 'business', 'plan', 'adobe', 'illustrator', 'adobe', 'photoshop', 'building', 'clientele', 'clienteling', 'computer', 'literacy', 'customer', 'service', 'design', 'garment', 'construction', 'microsoft', 'excel', 'microsoft', 'word', 'product', 'development', 'retail', 'math', 'retail', 'sales', 'sewing', 'styling', 'textiles', 'visual', 'merchandising']
BAG OF WORDS: [(6, 3), (10, 1), (14, 2), (16, 4), (18, 2), (21, 2), (24, 1), (26, 1), (27, 3), (38, 1), (40, 3), (55, 2), (57, 1), (62, 1), (66, 1), (68, 2), (120, 1), (123, 2), (124, 2), (144, 2), (163,

## Number of rows classified for each topic

In [74]:
classified_rows_bow = {};  classified_rows_tfidf = {};  s_time = time.time()
for i in range(num_topics):
    classified_rows_bow[str(i)] = []
    classified_rows_tfidf[str(i)] = []
# Bag of Words
for i in range(len(salaries_text_processed_sr)):
    scores = sorted(lda_model_bow[bow_corpus[i]], key=lambda tup: -1*tup[1])
    classified_rows_bow[str(scores[0][0])].append(i)
# TF-IDF
for i in range(len(salaries_text_processed_sr)):
    scores = sorted(lda_model_tfidf[bow_corpus[i]], key=lambda tup: -1*tup[1])
    classified_rows_tfidf[str(scores[0][0])].append(i)
print("-------- BoW  /  TF-IDF --------")
for i in range(num_topics):
    print("{:2} --> {:5}  /  {:5}".format(i, len(classified_rows_bow[str(i)]), len(classified_rows_tfidf[str(i)])))
    # print(i, '-->', len(classified_rows_bow[str(i)]), classified_rows[str(i)])
    # print(i, '-->', len(classified_rows_tfidf[str(i)]), classified_rows[str(i)])
print("TIME ELAPSED FOR TOPIC ASSIGNMENTS: {:.4f} SECONDS".format(time.time() - s_time))

-------- BoW  /  TF-IDF --------
 0 -->   188  /      0
 1 -->  2098  /    140
 2 -->   177  /      3
 3 -->  1233  /   1052
 4 -->  1032  /      0
 5 -->  1025  /      1
 6 -->  1435  /     40
 7 -->    10  /    667
 8 -->   113  /    686
 9 -->   275  /     22
10 -->   123  /     90
11 -->   824  /      0
12 -->   679  /    795
13 -->   700  /    302
14 -->  2340  /   3639
15 -->   279  /     62
16 -->   168  /      6
17 -->   792  /    934
18 -->  1349  /   5047
19 -->   817  /    174
20 -->  1977  /   4735
21 -->   622  /     59
22 -->   487  /      0
23 -->    39  /   1028
24 -->   772  /     72
TIME ELAPSED FOR TOPIC ASSIGNMENTS: 10.8642 SECONDS


## Prediction of the candidate's min-max salary & median based on the test case

In [79]:
# CHOOSE ONE OF THE SENIORITY LEVEL
# seniority = ''
seniority = 'senior '
# seniority = 'staff '
# seniority = 'senior staff '
# seniority = 'principal '
# seniority = 'tech lead '
# seniority = 'manager '
# seniority = 'senior manager '
# seniority = 'director '
# seniority = 'vice president '
# seniority = 'CEO '
# CHOOSE ONE OF THE TITLES
# title = seniority + 'software developer'
# title = seniority + 'software engineer'
# title = seniority + 'software test engineer'
title = seniority + 'data scientist'
# title = seniority + 'data analyst'
# title = seniority + 'machine learning engineer'
# title = seniority + 'business analyst'
# title = seniority + 'frontend developer'
# title = seniority + 'backend developer'
# title = seniority + 'devops engineer'
# title = seniority + 'test engineer'
# title = seniority + 'system test engineer'
# title = seniority + 'test automation engineer'
# title = seniority + 'quality assurance engineer'
# title = seniority + 'qa analyst'
# title = seniority + 'network engineer'
# title = seniority + 'it support engineer'
# title = seniority + 'security engineer'
# CHOOSE ONE OF THE MAIN SKILLS SET
# ms = 'java python spring'
# ms = 'hibernate java'
ms = 'agile angular2plus cplusplus elasticsearch hibernate manual testing mysql react spring'
# ms = 'data engineering java sql'
# ms = 'android programming'
# ms = 'dotnet csharp cplusplus data engineering java microsoft sql server mysql sql'
# ms = 'cplusplus python'
# ms = 'project management'
# ms = 'apache struts hibernate java jsf jsp oracle plsql spring'
# ms = 'dotnet angular2plus csharp docker elasticsearch microsoft sql server mongodb nodejs tsql'
# ms = 'data engineering microsoft sql server sql'
# ms = 'react user interface design'
# ms = 'csharp python data engineering java sql'
# ms = 'android programming apache kafka data engineering devops docker hibernate java jsf jsp mongodb nosql plsql postgresql python redis scrum spring sql'
# ms = 'computer networks information risk information security java python scala spring'
# ms = 'nextjs nodejs react'
# ms = 'docker elasticsearch google cloud platform grafana kubernetes mongodb mysql nginx postgresql python'
# ms = 'nodejs react'
# ms = 'business analysis data engineering java spring sql'
# ms = 'google cloud platform mysql nosql postgresql python selenium unity'
# ms = 'apache tomcat java'
# ms = 'angular2plus, devops mongoDB mysql nodejs user interface design'
# ms = 'agile cplusplus data Engineering java kubernetes scrum sql'
# ms = 'business analysis data engineering sql'
# ms = 'gaming'
# CHOOSE ONE OF THE SKILLS SET
# sk = 'java python spring framework'
# sk = 'eclipse hibernate java'
# sk = 'agile methodologies amazon web services aws angular bash cplusplus elasticsearch frontend development hibernate javascript jpa laravel linux liqubase manual testing mysql php reactjs rest apis spring boot typescript'
# sk = 'java object oriented programming oop sql teamwork'
# sk = 'android software development web development'
# sk = 'ajax aspdotnet mvc c csharp cplusplus eclipse html java java enterprise edition javascript jquery microsoft sql server mvc architecture mysql object oriented design oop software development software engineering sql subversion uml web services xml'
# sk = 'sdlc software project management testing'
# sk = 'ant axis bash eclipse hibernate j2ee java java enterprise edition javascript javase javaserver faces jsf jaxdashrpc jaxdashws jboss jboss application server jsf jsp log4j netbeans oracle oracle bpel oracle soa suite plsql primefaces project coordination sdlc servlets soa soap software design software development solution architecture spring framework struts technical leadership web applications web services weblogic websphere wsdl xml'
# sk = 'dotnet dotnet core amazon web hizmetleri aws angular angularjs aspdotnet csharp css docker elasticsearch html javascript jquery microsoft sql server mongodb nodejs oop rest tsql vuejs'
# sk = 'application security customer support enterprise support fiddler http infopath internet information services iis internet protocol ip javascript kali microsoft sql server oauth sql sql injection tcpip web application security web security windows xss zendesk'
# sk = 'javascript reactjs uiux'
# sk = 'c csharp cplusplus css eclipse extjs html html5 java javascript linux netbeans php software engineering sql sql db2 toad uml'
# sk = 'activemq android android development apache camel apache kafka camel cxf docker git go programming language hadoop hibernate hive jasper reports java jboss esb jenkins jpa jsf json jsp linux server maven mongodb multithreaded development nosql oracle sql developer plsql postgresql primefaces python redi restful webservices scrum servicemix soa socket programming spring boot spring framework sql ubuntu vuejs xml'
# sk = 'backend web development cybersecurity go programming language information security java linux network security python scala scripting software development spring boot'
# sk = 'amazon web services aws bootstrap d3js expressjs graphql headless cms html css jquery intrack javascript laravel mvc architecture nextjs nodejs nuxtjs php programming reactjs reduxjs research ruzzle testing unit testing vuejs vux webpack'
# sk = 'amazon web hizmetleri aws bash centos docker elasticsearch gcp grafana kubernetes linux mongodb mysql nginx postgresql python ubuntu vmware windows server zabbix'
# sk = 'laravel nodejs reactjs'
# sk = 'business analysis itil java project management professional sdlc spring framework sql unix'
# sk = 'css django flask google cloud functions google firebase html javascript mysql nosql postgresql python selenium unity3d'
# sk = 'apache tomcat java java ee'
# sk = 'adobe photoshop ajax angularjs apache cordova bitbucket bootstrap cascading style sheets css css dreamweaver git github gruntjs gulpjs heroku html html5 ionic framework javascript javascript beginner jquery jquery beginner meteor meteor ionic mongodb mysql nodejs photoshop php php beginner sass teamwork web design web development wordpress xhtml'
sk = 'agile methodologies algorithms c cplusplus cloud foundry computer vision design patterns digital signal processors git j2ee java javascript jira kubernetes linux machine learning matlab maven microsoft project multithreading object oriented design programming qt r scrum signal processing software design software development software engineering software testing sql subversion typescript uml visio'
# sk = 'business analysis business intelligence bi sql'
# sk = '3d 3d studio max video games'
# CHOOSE SUPERVISION SCORE
# sv_score = 10
sv_score = 6
# sv_score = 2
# CHOOSE ONE OF THE COUNTRY LOCATION
# country = 'United States'  # 1.000
# country = 'Canada'         # 0.769
# country = 'Israel'         # 0.763
country = 'Switzerland'    # 0.740
# country = 'Portugal'       # 0.635
# country = 'Italy'          # 0.621
# country = 'Spain'          # 0.612
# country = 'Uruguay'        # 0.602
# country = 'France'         # 0.596
# country = 'Finland'        # 0.556
# country = 'Czechia'        # 0.528
# country = 'Serbia'         # 0.476
# country = 'Panama'         # 0.450
# country = 'Turkey'         # 0.431
# country = 'India'          # 0.413
# country = 'Peru'           # 0.400
# country = 'Moldova'        # 0.350
# country = 'Georgia'        # 0.300
# ENTER WORKING YEAR VALUE (0-50 recommended)
wr_yr = 8
# ENTER THE NUMBER OF PEOPLE BEING MANAGED (0-2000 recommended)
pep_man = 0
# ENTER EDUCATION SCORE (5.0-9.9 recommended)
edu_score = 8.4
# ENTER LAST COMPANY SCORE (5.0-9.9 recommended)
comp_score = 8.1

# Select these 30 countries to predict the yearly salaries for each simultaneously
countries_for_salary_calc = ['United States', 'Turkey', 'Argentina', 'Australia', 'Austria', 'Bangladesh', 'Belarus',
    'Belgium', 'Brazil', 'Canada', 'China', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Iceland',
    'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway',
    'Pakistan', 'Philippines', 'Poland', 'Romania', 'Russia', 'Serbia', 'Singapore', 'Spain', 'South Korea',
    'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom', 'Vietnam']
# Show the BoW vector and classified topic (processed with both BoW LDA & TF-IDF LDA model)
print("### CANDIDATE INFO ###")
print("TITLE:", title)
if countries_coef[countries_coef['en_name'] == country]['salary_coefficient'].any().sum() == 0:
    print(">> COEFFICIENT NOT FOUND FOR {}!".format(country))
    country_coef = None
else:
    country_coef = countries_coef[countries_coef['en_name'] == country]['salary_coefficient'].values[0]
print("COUNTRY LOCATION: {} (COEFFICIENT = {})".format(country, country_coef))
print("MAIN SKILLS:", ms)
print("SKILLS:", sk)
print("WORKING YEAR:", wr_yr)
print("SUPERVISION SCORE:", sv_score)
print("EDUCATION SCORE:", edu_score)
print("LAST COMPANY SCORE:", comp_score)
print("# OF PEOPLE MANAGED:", pep_man)
candidate_text = title + ' ' + ms + ' ' + sk
bow_vector = dictionary.doc2bow(candidate_text.split(' '))
scores_bow = sorted(lda_model_bow[bow_vector], key=lambda tup: -1*tup[1])
scores_tfidf = sorted(lda_model_tfidf[bow_vector], key=lambda tup: -1*tup[1])
print("\nCONCATENATED TEXT:", candidate_text)
print("BOW VECTOR:", bow_vector)
print("-" * 35)
topic_words_bow = {}; topic_words_tfidf = {}
print("CLASSIFIED TOPIC BoW >>>", scores_bow[0][0])
# print(scores_bow)
# print(scores_bow[0][0], '|', lda_model_bow.print_topic(scores_bow[0][0], 10))
print("SCORE                | TOPIC")
for index, score in scores_bow:
    topic_words_bow['TOPIC '+str(index)] = lda_model_bow.print_topic(index, 10)
    print("{:<20} | {:2} --> {}".format(score, index, lda_model_bow.print_topic(index, 5)))
# print(topic_words_bow)
print("-" * 35)
print("CLASSIFIED TOPIC TF-IDF >>>", scores_tfidf[0][0])
# print(scores_tfidf)
# print(scores_tfidf[0][0], '|', lda_model_tfidf.print_topic(scores_tfidf[0][0], 10))
print("SCORE                | TOPIC")
for index, score in scores_tfidf:
    topic_words_tfidf['TOPIC '+str(index)] = lda_model_tfidf.print_topic(index, 10)
    print("{:<20} | {:2} --> {}".format(score, index, lda_model_tfidf.print_topic(index, 5)))
# print(topic_words_tfidf)

### CANDIDATE INFO ###
TITLE: senior data scientist
COUNTRY LOCATION: Switzerland (COEFFICIENT = 0.74)
MAIN SKILLS: agile angular2plus cplusplus elasticsearch hibernate manual testing mysql react spring
SKILLS: agile methodologies algorithms c cplusplus cloud foundry computer vision design patterns digital signal processors git j2ee java javascript jira kubernetes linux machine learning matlab maven microsoft project multithreading object oriented design programming qt r scrum signal processing software design software development software engineering software testing sql subversion typescript uml visio
WORKING YEAR: 8
SUPERVISION SCORE: 6
EDUCATION SCORE: 8.4
LAST COMPANY SCORE: 8.1
# OF PEOPLE MANAGED: 0

CONCATENATED TEXT: senior data scientist agile angular2plus cplusplus elasticsearch hibernate manual testing mysql react spring agile methodologies algorithms c cplusplus cloud foundry computer vision design patterns digital signal processors git j2ee java javascript jira kubernetes

In [81]:
def predict_candidate_salary(tag, scores, classified_rows):
    print("#### {} ####".format(tag))
    # Inspect the candidate's title and determine the title coefficient
    # if 'director' in title:           title_coef = 2.4
    # elif 'senior manager' in title:   title_coef = 1.8
    # elif 'manager' in title:          title_coef = 1.6
    # elif 'lead' in title:             title_coef = 1.3
    # else:   title_coef = 1.0
    # print("TITLE COEFFICIENT: {}  (TITLE = {})".format(title_coef, title))
    # Inspect the candidate's supervision score and determine the supervision coefficient
    if sv_score == 10:     supervision_coef = 1.0   # 2.2
    elif sv_score == 6:    supervision_coef = 1.0   # 1.6
    elif sv_score == 2:    supervision_coef = 1.0
    else:    supervision_coef = 1.0
    print("SUPERVISION COEFFICIENT: {} (SUPERVISION SCORE = {})".format(supervision_coef, sv_score))
    # Inspect the number of people being managed and determine the people management coefficient
    if pep_man >= 1001:   pep_man_coef = 5.5
    elif pep_man >= 501 and pep_man <= 1000:   pep_man_coef = 4.8
    elif pep_man >= 101 and pep_man <= 500:    pep_man_coef = 3.6
    elif pep_man >= 26 and pep_man <= 100:     pep_man_coef = 2.8
    elif pep_man >= 11 and pep_man <= 25:      pep_man_coef = 1.6
    elif pep_man >= 1 and pep_man <= 10:       pep_man_coef = 1.3
    else:   pep_man_coef = 1.0
    print("PEOPLE MANAGEMENT COEFFICIENT: {}  (PEOPLE MANAGED = {})".format(pep_man_coef, pep_man))
    # Inspect the candidate's education score and determine the education coefficient
    if edu_score >= 9.5:   edu_coef = 2.0
    elif edu_score >= 9.0 and edu_score <= 9.4:   edu_coef = 1.7
    elif edu_score >= 8.0 and edu_score <= 8.9:   edu_coef = 1.4
    elif edu_score >= 7.0 and edu_score <= 7.9:   edu_coef = 1.2
    else:   edu_coef = 1.0
    print("EDUCATION COEFFICIENT: {}  (EDUCAITON SCORE = {})".format(edu_coef, edu_score))
    # Inspect the candidate's last company score and determine the company coefficient
    if comp_score >= 9.0:   comp_coef = 2.5
    elif comp_score >= 8.0 and comp_score <= 8.9:   comp_coef = 2.0
    else:   comp_coef = 1.0
    print("COMPANY COEFFICIENT: {}  (LAST COMPANY SCORE = {})".format(comp_coef, comp_score))
    # From these determined education and company coefs and the candidate's working year, get the weighted coefficient
    if wr_yr < 2:   edu_weight = 0.5;   comp_weight = 0.5
    elif wr_yr >= 2 and wr_yr <= 5:   edu_weight = 0.3;   comp_weight = 0.7
    elif wr_yr >= 6 and wr_yr <= 10:   edu_weight = 0.2;   comp_weight = 0.8
    elif wr_yr >= 11:   edu_weight = 0.1;   comp_weight = 0.9
    weighted_coef = edu_weight * edu_coef + comp_weight * comp_coef
    print("WEIGHTED COEFFICIENT: {} * {} + {} * {} = {:4.2f}  (WORKING YEAR = {})".format(
        edu_weight, edu_coef, comp_weight, comp_coef, weighted_coef, wr_yr))
    print("COUNTRY COEFFICIENT: {} ({})".format(country_coef, country))
    
    # After all these calculations, let's obtain the salary coefficient to be applied after the prediction
    # However, the important condition is that there must be valid country coefficient, otherwise it will not continue!
    if country_coef == None:
        print("\nCANNOT PREDICT MIN-MAX SALARY AND MEDIAN! (REASON: COUNTRY COEFFICIENT DOES NOT EXIST)")
        return
    elif pep_man_coef != 1.0:   salary_coef = pep_man_coef * weighted_coef * country_coef
    else:   salary_coef = supervision_coef * weighted_coef * country_coef
    print(">>> SALARY COEFFICIENT: {:.3f}".format(salary_coef))
    
    # Get the classified topic which is needed for all the operations below
    classified_topic = scores[0][0]
    print("\n>>> CLASSIFIED TOPIC:", classified_topic)
    # Create a subset of salary data for those from the chosen classified topic
    years_salaries_tpc = salaries.iloc[classified_rows[str(classified_topic)]] \
                                          [['working_year','amount_usd']].reset_index()
    years_salaries_count = {};   years_salaries = {}
    # Determine the categories of working years and assign the min-max experience years
    if wr_yr >= 15:   
        years_range = ['10-14', '15+'];            minExp = 10;    maxExp = 50
    elif wr_yr >= 10 and wr_yr <= 14:  
        years_range = ['7-9', '10-14', '15+'];     minExp = 7;    maxExp = 50
    elif wr_yr >= 7 and wr_yr <= 9:   
        years_range = ['4-6', '7-9', '10-14'];     minExp = 4;    maxExp = 14
    elif wr_yr >= 4 and wr_yr <= 6:     
        years_range = ['1-3', '4-6', '7-9'];       minExp = 1;    maxExp = 9
    elif wr_yr >= 1 and wr_yr <=3:
        years_range = ['0', '1-3', '4-6'];         minExp = 0;    maxExp = 6
    else:    
        years_range = ['0', '1-3'];                minExp = 0;    maxExp = 3
    print("MINIMUM-MAXIMUM YEAR EXPERIENCE RANGE: {}-{}".format(minExp, maxExp))
    
    # Determine the salary interval thresholds w.r.t. the country coefficient
    salary_intrv_thr = [int(country_coef * intrv) for intrv in [100000, 140000, 180000, 220000, 260000, 300000, 340000]]
    print("SALARY INTERVAL THRESHOLDS:", salary_intrv_thr)
    # Then, construct their interval labels as yearly
    salary_intrv_label_y = []
    for i in range(len(salary_intrv_thr)):
        if i == 0:
            salary_intrv_label_y.append('<'+str(salary_intrv_thr[i] // 1000)+'K')
            salary_intrv_label_y.append(str(salary_intrv_thr[i] // 1000)+'K-'+str(salary_intrv_thr[i+1] // 1000)+'K')
        elif i == len(salary_intrv_thr)-1:
            salary_intrv_label_y.append('>='+str(salary_intrv_thr[i] // 1000)+'K')
        else:
            salary_intrv_label_y.append(str(salary_intrv_thr[i] // 1000)+'K-'+str(salary_intrv_thr[i+1] // 1000)+'K')
    print("SALARY INTERVAL LABELS (Y):", salary_intrv_label_y)
    # Moreover, do the same construction as monthly
    salary_intrv_label_m = []
    for i in range(len(salary_intrv_thr)):
        if i == 0:
            salary_intrv_label_m.append('<'+str(salary_intrv_thr[i] // 1000 // 12)+'K')
            salary_intrv_label_m.append(str(salary_intrv_thr[i] // 1000 // 12)+'K-'
                                        +str(salary_intrv_thr[i+1] // 1000 // 12)+'K')
        elif i == len(salary_intrv_thr)-1:
            salary_intrv_label_m.append('>='+str(salary_intrv_thr[i] // 1000 // 12)+'K')
        else:
            salary_intrv_label_m.append(str(salary_intrv_thr[i] // 1000 // 12)+'K-'
                                        +str(salary_intrv_thr[i+1] // 1000 // 12)+'K')
    print("SALARY INTERVAL LABELS (M):", salary_intrv_label_m)
    
    # Place all these salaries of the subset into the appropriate lists w.r.t. their categories of working years 
    # and salary intervals. This is done for individual working years category and the combination of all
    for year in years_range:
        years_salaries_count[year] = [0 for _ in range(8)]
        years_salaries[year] = [[] for _ in range(8)]
    years_salaries_comb = [[] for _ in range(8)]
    years_salaries_whole = []
    for i in range(len(years_salaries_tpc)):
        year, salary = years_salaries_tpc.loc[i, 'working_year'], years_salaries_tpc.loc[i, 'amount_usd']
        # Determine the condition of working years for correct selection
        # Pay attention to both the candidate's and the related row's working year together
        # If the related row's working year value is not in one of the categories, skip to the next one
        if wr_yr >= 15 and year >= 10:
            year_cond = '10-14' if year >= 10 and year <= 14 else '15+'
        elif (wr_yr >= 10 and wr_yr <= 14) and year >= 7:
            year_cond = '7-9' if year >= 7 and year <= 9 else '10-14' if year >= 10 and year <= 14 else '15+'
        elif (wr_yr >= 7 and wr_yr <= 9) and (year >= 4 and year <= 14):
            year_cond = '4-6' if year >= 4 and year <= 6 else '7-9' if year >= 7 and year <= 9 else '10-14' 
        elif (wr_yr >= 4 and wr_yr <= 6) and (year >= 1 and year <= 9):
            year_cond = '1-3' if year >= 1 and year <= 3 else '4-6' if year >= 4 and year <= 6 else '7-9'
        elif (wr_yr >= 1 and wr_yr <= 3) and year <= 6:
            year_cond = '0' if year == 0 else '1-3' if year >= 1 and year <= 3 else '4-6'
        elif wr_yr == 0 and year <= 3:
            year_cond = '0' if year == 0 else '1-3'
        else:
            continue
        # salaries_cond = 0 if salary < salary_intrv_thr[0] \
        #     else 1 if salary >= salary_intrv_thr[0] and salary < salary_intrv_thr[1] \
        #     else 2 if salary >= salary_intrv_thr[1] and salary < salary_intrv_thr[2] \
        #     else 3 if salary >= salary_intrv_thr[2] and salary < salary_intrv_thr[3] \
        #     else 4 if salary >= salary_intrv_thr[3] and salary < salary_intrv_thr[4] \
        #     else 5 if salary >= salary_intrv_thr[4] and salary < salary_intrv_thr[5] \
        #     else 6 if salary >= salary_intrv_thr[5] and salary < salary_intrv_thr[6] else 7
        salaries_cond = 0 if salary < 100000 else 1 if salary >= 100000 and salary < 140000 \
            else 2 if salary >= 140000 and salary < 180000 else 3 if salary >= 180000 and salary < 220000 \
            else 4 if salary >= 220000 and salary < 260000 else 5 if salary >= 260000 and salary < 300000 \
            else 6 if salary >= 300000 and salary < 340000 else 7
        # Count the salary according to the conditions set above and append it to the appropriate salary interval list
        years_salaries_count[year_cond][salaries_cond] += 1
        years_salaries[year_cond][salaries_cond].append(salary)
        years_salaries_comb[salaries_cond].append(salary)
        years_salaries_whole.append(salary)
    
    # Make a dataframe of the counted salaries w.r.t. working year categories and salary intervals
    years_salaries_df = pd.DataFrame(years_salaries_count, index=salary_intrv_label_y)
    print(years_salaries_df)
    # Get the total number of salaries included in the counts and lists (also get results by years and intervals)
    total_salaries = years_salaries_df.sum().sum()
    total_salaries_years = years_salaries_df.sum().values
    total_salaries_intrv = years_salaries_df.sum(axis=1).values
    print("{} SALARIES AVAILABLE | BY YEARS: {} | BY INTERVALS: {}\n".format(
          total_salaries, total_salaries_years, total_salaries_intrv))
    # Get the percentage distribution of all defined salary intervals for all included salaries
    if total_salaries != 0:
        print("SALARY INTERVALS PERC:", 
              np.around(years_salaries_df.sum(axis=1).values / total_salaries * 100, 3).tolist(), "\n")
    # Show the salaries appended to each interval salary list
    # And calculate the percentage distribution for each working year category, then reveal them to the output
    years_salaries_prob = [];   years_salaries_prob_comb = [];  i = 0
    for year in years_range:
        # print("SALARIES ({}): {}".format(year, years_salaries[year]))  # Be aware that the salary list might be too long!
        if sum(years_salaries_count[year]) != 0:
            years_salaries_prob.append([round(val / sum(years_salaries_count[year]) * 100, 3) 
                                        for val in years_salaries_count[year]])
        else:
            years_salaries_prob.append([0.0 for _ in range(8)])  
        print("PROB ({}): {}\n".format(year, years_salaries_prob[i]))
        i += 1
    # Do the same thing above for the combined salaries of each interval regardless of working year categories
    # print("SALARIES COMBINED: {}".format(years_salaries_comb))  # Be aware that the salary list might be too long!
    for i in range(len(years_salaries_comb)):
        years_salaries_prob_comb.append(round(len(years_salaries_comb[i]) / years_salaries_df.sum().sum() * 100, 3)
                                        if len(years_salaries_comb[i]) != 0 else 0.0)
    print("PROB COMBINED: {}".format(years_salaries_prob_comb))
    
    # Now calculate the predicted salary for each working year category (there might be two or three values normally)
    # However, any working year category that does not have any salary data will not be included
    years_salaries_calc = [];  j = 0
    for year in years_range:
        calc_value = 0.0
        for i in range(len(years_salaries[year])):
            if len(years_salaries[year][i]) == 0:
                continue
            # print(years_salaries[year][i])
            # Value is added by getting the average of salaries in the interval, multiplied by its percentage in the dist.
            calc_value += np.array(years_salaries[year][i]).mean() * (years_salaries_prob[j][i] / 100)
        if calc_value != 0.0:
            years_salaries_calc.append(round(calc_value,4))
        j += 1
    print("\nPREDICTED SALARIES OF EACH YEAR CATEGORY:", years_salaries_calc)
    
    # The same operation is also carried out with the combined variant
    years_salaries_calc_comb = 0.0
    for i in range(len(years_salaries_comb)):
        if len(years_salaries_comb[i]) == 0:
            continue
        years_salaries_calc_comb += np.array(years_salaries_comb[i]).mean() * (years_salaries_prob_comb[i] / 100)
    print("PREDICTED SALARY OF COMBINED:", round(years_salaries_calc_comb,4), "\n")
    print("{} SALARIES WERE INCLUDED FOR PREDICTION AND COEFFICIENT OF {:.3f} WAS APPLIED AFTERWARDS".format(
        total_salaries, salary_coef))
    
    # Now is the time to predict the salary and its min-max range for the candidate!
    # Af this step, salary prediction is performed only if the total number of salaries in the range is at least 15
    if total_salaries < 15:
        print("CANNOT PREDICT MIN-MAX SALARY AND MEDIAN! (REASON: NOT ENOUGH NUMBER OF SALARIES [< 15])")
    else:
        print("\n### PREDICTED MIN-MAX SALARY RESULTS ###")
        # Option 1: Get the minimum and maximum value from the predicted salaries of each year category
        print("OPTION 1 - YEAR CATEGORY MIN-MAX >>  {:10.3f} - {:10.3f}".format(
              round(min(years_salaries_calc) * salary_coef,-3), round(max(years_salaries_calc) * salary_coef,-3)))
        # Option 2: Find the average of the predicted salaries of each year category
        #           And get 90% and 110% of the average to assess them as min-max values 
        years_salaries_avg = np.array(years_salaries_calc).mean()
        print("OPTION 2 - YEAR CATEGORY AVERAGE >>  {:10.3f} - {:10.3f}".format(
              round(years_salaries_avg * 0.9 * salary_coef,-3), round(years_salaries_avg * 1.1 * salary_coef,-3)))
        # Option 3: Get 90% and 110% of the combined variant of predicted salary to assess them as min-max values
        print("OPTION 3 - YEAR CATEGORY COMBINED AVERAGE >>  {:10.3f} - {:10.3f}".format(
              round(years_salaries_calc_comb * 0.9 * salary_coef,-3), 
              round(years_salaries_calc_comb * 1.1 * salary_coef,-3)))
        
        # Additionally, determine the median value of salaries with three different options applied below
        # The numbers in parentheses indicate the total number of salaries included for finding the median
        print("\n### MEDIAN RESULTS ###")
        # Option 1: All salaries from the classified topic
        print("OPTION 1 - ALL SALARIES IN CHOSEN TOPIC({}) >>  {:.3f}".format(
              len(years_salaries_tpc), round(np.median(years_salaries_tpc['amount_usd']) * salary_coef,-3)))
        # Option 2: All salaries included in working year categories
        ys_whole = np.array(years_salaries_whole)
        print("OPTION 2 - ALL SALARIES IN WORKING YEARS RANGE({}) >>  {:.3f}".format(
              len(ys_whole), round(np.median(ys_whole) * salary_coef,-3)))
        # Option 3: All salaries that are within the min-max range of predicted salary
        # Note: Option 2 of salary prediction above is considered for the range
        ys_whole_range = ys_whole[(ys_whole > years_salaries_avg * 0.9) & (ys_whole < years_salaries_avg * 1.1)]
        print("OPTION 3 - ALL SALARIES IN PREDICTED MIN-MAX({}) >>  {:.3f}".format(
              len(ys_whole_range), round(np.median(ys_whole_range) * salary_coef,-3)))
        # print(np.sort(ys_whole_range))
        
        # Lastly, perform the prediction of yearly salaries by 30 different countries at once
        # Note that the 2nd option of salary results are being used
        print("\n### SALARIES BY COUNTRIES ###")
        years_salaries_avg = np.array(years_salaries_calc).mean()
        countries_salaries = {};   c = 0
        for country_name in countries_for_salary_calc:
            coef = countries_coef[countries_coef['en_name'] == country_name]['salary_coefficient'].values[0]
            cntr_slr = round(years_salaries_avg * supervision_coef * weighted_coef * coef,-2)
            countries_salaries[country_name] = cntr_slr
        for country_name in countries_salaries.items():
            c += 1
            if c % 3 == 0:   print("{:16} -> {:9.1f}".format(country_name[0], country_name[1]))
            else:   print("{:15} -> {:9.1f}".format(country_name[0], country_name[1]), end=" | ")
        print()

print("### CANDIDATE INFO ###")
print("TITLE:", title)
print("COUNTRY:", country)
print("MAIN SKILLS:", ms)
print("SKILLS:", sk)
print("WORKING YEAR:", wr_yr)
print("EDUCATION SCORE:", edu_score)
print("LAST COMPANY SCORE:", comp_score)
print("# OF PEOPLE MANAGED:", pep_man)
print("\nCONCATENATED TEXT:", candidate_text)
print("BOW VECTOR:", bow_vector)
print("-" * 95)
predict_candidate_salary("BoW", scores_bow, classified_rows_bow)
print("-" * 95)
predict_candidate_salary("TF-IDF", scores_tfidf, classified_rows_tfidf)
print('-' * 95)

### CANDIDATE INFO ###
TITLE: senior data scientist
COUNTRY: Switzerland
MAIN SKILLS: agile angular2plus cplusplus elasticsearch hibernate manual testing mysql react spring
SKILLS: agile methodologies algorithms c cplusplus cloud foundry computer vision design patterns digital signal processors git j2ee java javascript jira kubernetes linux machine learning matlab maven microsoft project multithreading object oriented design programming qt r scrum signal processing software design software development software engineering software testing sql subversion typescript uml visio
WORKING YEAR: 8
EDUCATION SCORE: 8.4
LAST COMPANY SCORE: 8.1
# OF PEOPLE MANAGED: 0

CONCATENATED TEXT: senior data scientist agile angular2plus cplusplus elasticsearch hibernate manual testing mysql react spring agile methodologies algorithms c cplusplus cloud foundry computer vision design patterns digital signal processors git j2ee java javascript jira kubernetes linux machine learning matlab maven microsoft proj